# Recovery: GWB (13/3, -14) + band noise LOFAR (-12.5), stessa realizzazione delle sim OFF e A125

Dati da `Simulations_GWB_A140_su_bn_A125_sv44.ipynb`: il GWB e stato **sommato** alle simulazioni del notebook 21,
quindi red noise, DM, scattering e band noise sono identici a quelli di OFF e A125. Le differenze fra le posteriori
sono dovute solo a GWB e band noise, non alla realizzazione.

| dataset | dati | 6 par | 8 par |
|---|---|---|---|
| OFF (niente) | `sim_data_noise_bn_OFF_sv44_50componenti/` | `test_sim_bn_OFF_50comp/` (nb 21) | |
| solo band -12.5 | `sim_data_noise_bn_A125_sv44_50componenti/` | `test_sim_bn_A125_50comp/` (nb 21) | `test_sim_bn_A125_bnmodel/` (nb 21) |
| solo GWB -14 | `sim_data_noise_bn_OFF_GWB_A140_sv44_50componenti/` | `test_sim_bn_OFF_GWB_6par/` (**qui**) | |
| GWB + band | `sim_data_noise_bn_A125_GWB_A140_sv44_50componenti/` | `test_sim_bn_A125_GWB_6par/` (**qui**) | `test_sim_bn_A125_GWB_8par/` (**qui**) |
| dati reali | | `test_real_sv44/` (nb 10) | `test_real_bn_bnmodel/` (nb 20) |

Tre run nuove, ciascuna con il suo flag.

In [ ]:
import numpy as np
import os, glob, json
import pickle

import sys

# this is the path to the discovery folder
sys.path.insert(0, "./")
import discovery_new as ds

import matplotlib.pyplot as plt
import scipy
import jax.numpy as jnp

from scipy.stats import uniform
import re
from PTMCMCSampler.PTMCMCSampler import PTSampler as ptmcmc
from scipy.stats import uniform
import jax

import time
import emcee
import corner

In [ ]:
##### Cell for the prior distributions ###################################################################################

my_priordict = {}
my_priordict["(.*_)?sv_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?sv_gp_alpha.*"] = [0, 10]
my_priordict["(.*_)?sv_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_log10_A.*"] = [-10, 1]
my_priordict["(.*_)?sw_gamma.*"] = [-5, 6]
my_priordict["(.*_)?dm_gp_gamma.*"] = [0, 7]
my_priordict["(.*_)?dm_gp_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?red_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?red_noise_log10_A.*"] = [-20, -11]
my_priordict["(.*_)?sw_n_earth.*"] = [0, 30]
# band noise: stesse prior degli altri processi power-law
my_priordict["(.*_)?band_noise_gamma.*"] = [0, 7]
my_priordict["(.*_)?band_noise_log10_A.*"] = [-20, -11]


def build_priors(param_names, priordict):
    """
    Build a {param_name: scipy dist} dict with uniform prior distributions in the range
    """
    priors = {}
    unmatched = []
    for name in param_names:
        matched = False
        for pattern, (lo, hi) in priordict.items():
            if re.fullmatch(pattern, name):
                priors[name] = uniform(loc=lo, scale=hi - lo)
                matched = True
                break
        if not matched:
            unmatched.append(name)

    if unmatched:
        print(f"Warning: no prior found for {len(unmatched)} parameters, "
              f"falling back to standard normal:\n  {unmatched}")
    return priors

## Dati simulati: GWB a -14 + band noise a -12.5

In [ ]:
gwbbn125_psr = ds.pulsar.Pulsar.read_feather('sim_data_noise_bn_A125_GWB_A140_sv44_50componenti/J1640+2224')   # GWB a -14 + band noise a -12.5
print('nome     :', gwbbn125_psr.name)
print('N_TOA    :', len(gwbbn125_psr.toas))
print('shape M  :', gwbbn125_psr.Mmat.shape)

## GWB + band, modello a 6 par

In [ ]:
# modello a 6 par: quello standard, senza band noise
model_gwbbn125_6par = [gwbbn125_psr.residuals]
model_gwbbn125_6par += [ds.makegp_timing(gwbbn125_psr, svd=True)]
model_gwbbn125_6par += [ds.makenoise_measurement(gwbbn125_psr, gwbbn125_psr.noisedict, tnequad=True, ecorr=True)]
model_gwbbn125_6par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_red_components'],
                                 T=ds.getspan(gwbbn125_psr), name='red_noise'))
model_gwbbn125_6par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_dm_components'],
                                 T=ds.getspan(gwbbn125_psr), name='dm_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_gwbbn125_6par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_chrom_components'],
                                 T=ds.getspan(gwbbn125_psr), name='sv_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=4.4, tndm=False)))

logL_gwbbn125_6par = ds.PulsarLikelihood(model_gwbbn125_6par).logL
logL_gwbbn125_6par.params

In [ ]:
def params_dict_from_array_gwbbn125_6par(x):
    x = np.asarray(x)
    return dict(zip(logL_gwbbn125_6par.params, x))

priors_gwbbn125_6par = build_priors(logL_gwbbn125_6par.params, my_priordict)

def log_prior_gwbbn125_6par(x):
    params = params_dict_from_array_gwbbn125_6par(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_gwbbn125_6par:
            logp += priors_gwbbn125_6par[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_gwbbn125_6par():
    return np.array([priors_gwbbn125_6par[name].rvs() for name in logL_gwbbn125_6par.params])

x0_gwbbn125_6par = prior_draw_gwbbn125_6par()

In [ ]:
@jax.jit
def _jit_logL_gwbbn125_6par(params_array):
    return logL_gwbbn125_6par(dict(zip(logL_gwbbn125_6par.params, params_array)))

def log_likelihood_gwbbn125_6par(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_gwbbn125_6par(jnp.array(x))))

print('logL iniziale =', log_likelihood_gwbbn125_6par(x0_gwbbn125_6par))

In [ ]:
sampler_gwbbn125_6par = ptmcmc(
    ndim=len(logL_gwbbn125_6par.params),
    logl=log_likelihood_gwbbn125_6par,
    logp=log_prior_gwbbn125_6par,
    cov=np.eye(len(logL_gwbbn125_6par.params)) * 0.001,
    outDir='./test_sim_bn_A125_GWB_6par/',
    resume=True
)

In [ ]:
ESEGUI_SAMPLING_GWBBN125_6PAR = False   # True solo per lanciare davvero il campionamento

if ESEGUI_SAMPLING_GWBBN125_6PAR:
    print("Starting sampling (GWB -14 + band -12.5, modello a 6 par)...")
    sampler_gwbbn125_6par.sample(x0_gwbbn125_6par, Niter=1000000, thin=10)

In [ ]:
chain_gwbbn125_6par = np.loadtxt('./test_sim_bn_A125_GWB_6par/chain_1.txt')[:, :-4]
chain_gwbbn125_6par = chain_gwbbn125_6par[len(chain_gwbbn125_6par)//4:]
chain_gwbbn125_6par.shape

In [ ]:
tau_gwbbn125_6par = emcee.autocorr.integrated_time(chain_gwbbn125_6par, has_walkers=False, quiet=True)
print('tau:', tau_gwbbn125_6par)
print('Effective length =', int(len(chain_gwbbn125_6par) / tau_gwbbn125_6par.max()))

## GWB + band, modello a 8 par

In [ ]:
# modello a 8 par: i 6 standard + band noise sui backend LOFAR
model_gwbbn125_8par = [gwbbn125_psr.residuals]
model_gwbbn125_8par += [ds.makegp_timing(gwbbn125_psr, svd=True)]
model_gwbbn125_8par += [ds.makenoise_measurement(gwbbn125_psr, gwbbn125_psr.noisedict, tnequad=True, ecorr=True)]
model_gwbbn125_8par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_red_components'],
                                 T=ds.getspan(gwbbn125_psr), name='red_noise'))
model_gwbbn125_8par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_dm_components'],
                                 T=ds.getspan(gwbbn125_psr), name='dm_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_gwbbn125_8par.append(ds.makegp_fourier(gwbbn125_psr, ds.powerlaw, gwbbn125_psr.noisedict[gwbbn125_psr.name + '_chrom_components'],
                                 T=ds.getspan(gwbbn125_psr), name='sv_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=4.4, tndm=False)))
model_gwbbn125_8par.append(ds.makegp_fourier_select(gwbbn125_psr, ds.powerlaw, components=50, T=ds.getspan(gwbbn125_psr),
                                        name='band_noise', selection=ds.lofar_backends))

logL_gwbbn125_8par = ds.PulsarLikelihood(model_gwbbn125_8par).logL
logL_gwbbn125_8par.params

In [ ]:
def params_dict_from_array_gwbbn125_8par(x):
    x = np.asarray(x)
    return dict(zip(logL_gwbbn125_8par.params, x))

priors_gwbbn125_8par = build_priors(logL_gwbbn125_8par.params, my_priordict)

def log_prior_gwbbn125_8par(x):
    params = params_dict_from_array_gwbbn125_8par(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_gwbbn125_8par:
            logp += priors_gwbbn125_8par[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_gwbbn125_8par():
    return np.array([priors_gwbbn125_8par[name].rvs() for name in logL_gwbbn125_8par.params])

x0_gwbbn125_8par = prior_draw_gwbbn125_8par()

In [ ]:
@jax.jit
def _jit_logL_gwbbn125_8par(params_array):
    return logL_gwbbn125_8par(dict(zip(logL_gwbbn125_8par.params, params_array)))

def log_likelihood_gwbbn125_8par(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_gwbbn125_8par(jnp.array(x))))

print('logL iniziale =', log_likelihood_gwbbn125_8par(x0_gwbbn125_8par))

In [ ]:
sampler_gwbbn125_8par = ptmcmc(
    ndim=len(logL_gwbbn125_8par.params),
    logl=log_likelihood_gwbbn125_8par,
    logp=log_prior_gwbbn125_8par,
    cov=np.eye(len(logL_gwbbn125_8par.params)) * 0.001,
    outDir='./test_sim_bn_A125_GWB_8par/',
    resume=True
)

In [ ]:
ESEGUI_SAMPLING_GWBBN125_8PAR = False   # True solo per lanciare davvero il campionamento

if ESEGUI_SAMPLING_GWBBN125_8PAR:
    print("Starting sampling (GWB -14 + band -12.5, modello a 8 par)...")
    sampler_gwbbn125_8par.sample(x0_gwbbn125_8par, Niter=1000000, thin=10)

In [ ]:
chain_gwbbn125_8par = np.loadtxt('./test_sim_bn_A125_GWB_8par/chain_1.txt')[:, :-4]
chain_gwbbn125_8par = chain_gwbbn125_8par[len(chain_gwbbn125_8par)//4:]
chain_gwbbn125_8par.shape

In [ ]:
tau_gwbbn125_8par = emcee.autocorr.integrated_time(chain_gwbbn125_8par, has_walkers=False, quiet=True)
print('tau:', tau_gwbbn125_8par)
print('Effective length =', int(len(chain_gwbbn125_8par) / tau_gwbbn125_8par.max()))

## Solo GWB (stessa realizzazione), modello a 6 par

In [ ]:
gwboff_psr = ds.pulsar.Pulsar.read_feather('sim_data_noise_bn_OFF_GWB_A140_sv44_50componenti/J1640+2224')   # solo GWB a -14, band noise spento
print('nome     :', gwboff_psr.name)
print('N_TOA    :', len(gwboff_psr.toas))
print('shape M  :', gwboff_psr.Mmat.shape)

In [ ]:
# modello a 6 par: quello standard, senza band noise
model_gwboff_6par = [gwboff_psr.residuals]
model_gwboff_6par += [ds.makegp_timing(gwboff_psr, svd=True)]
model_gwboff_6par += [ds.makenoise_measurement(gwboff_psr, gwboff_psr.noisedict, tnequad=True, ecorr=True)]
model_gwboff_6par.append(ds.makegp_fourier(gwboff_psr, ds.powerlaw, gwboff_psr.noisedict[gwboff_psr.name + '_red_components'],
                                 T=ds.getspan(gwboff_psr), name='red_noise'))
model_gwboff_6par.append(ds.makegp_fourier(gwboff_psr, ds.powerlaw, gwboff_psr.noisedict[gwboff_psr.name + '_dm_components'],
                                 T=ds.getspan(gwboff_psr), name='dm_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=2.0, tndm=False)))
model_gwboff_6par.append(ds.makegp_fourier(gwboff_psr, ds.powerlaw, gwboff_psr.noisedict[gwboff_psr.name + '_chrom_components'],
                                 T=ds.getspan(gwboff_psr), name='sv_gp',
                                 fourierbasis=ds.make_dmfourierbasis(alpha=4.4, tndm=False)))

logL_gwboff_6par = ds.PulsarLikelihood(model_gwboff_6par).logL
logL_gwboff_6par.params

In [ ]:
def params_dict_from_array_gwboff_6par(x):
    x = np.asarray(x)
    return dict(zip(logL_gwboff_6par.params, x))

priors_gwboff_6par = build_priors(logL_gwboff_6par.params, my_priordict)

def log_prior_gwboff_6par(x):
    params = params_dict_from_array_gwboff_6par(x)
    logp = 0.0
    for name, value in params.items():
        if name in priors_gwboff_6par:
            logp += priors_gwboff_6par[name].logpdf(value)
        else:
            logp += -0.5 * np.sum(np.asarray(value)**2)
    return logp

def prior_draw_gwboff_6par():
    return np.array([priors_gwboff_6par[name].rvs() for name in logL_gwboff_6par.params])

x0_gwboff_6par = prior_draw_gwboff_6par()

In [ ]:
@jax.jit
def _jit_logL_gwboff_6par(params_array):
    return logL_gwboff_6par(dict(zip(logL_gwboff_6par.params, params_array)))

def log_likelihood_gwboff_6par(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.array(_jit_logL_gwboff_6par(jnp.array(x))))

print('logL iniziale =', log_likelihood_gwboff_6par(x0_gwboff_6par))

In [ ]:
sampler_gwboff_6par = ptmcmc(
    ndim=len(logL_gwboff_6par.params),
    logl=log_likelihood_gwboff_6par,
    logp=log_prior_gwboff_6par,
    cov=np.eye(len(logL_gwboff_6par.params)) * 0.001,
    outDir='./test_sim_bn_OFF_GWB_6par/',
    resume=True
)

In [ ]:
ESEGUI_SAMPLING_GWBOFF_6PAR = False   # True solo per lanciare davvero il campionamento

if ESEGUI_SAMPLING_GWBOFF_6PAR:
    print("Starting sampling (solo GWB -14, modello a 6 par)...")
    sampler_gwboff_6par.sample(x0_gwboff_6par, Niter=1000000, thin=10)

In [ ]:
chain_gwboff_6par = np.loadtxt('./test_sim_bn_OFF_GWB_6par/chain_1.txt')[:, :-4]
chain_gwboff_6par = chain_gwboff_6par[len(chain_gwboff_6par)//4:]
chain_gwboff_6par.shape

In [ ]:
tau_gwboff_6par = emcee.autocorr.integrated_time(chain_gwboff_6par, has_walkers=False, quiet=True)
print('tau:', tau_gwboff_6par)
print('Effective length =', int(len(chain_gwboff_6par) / tau_gwboff_6par.max()))

## Catene di confronto (gia esistenti)

In [ ]:
chain_off       = np.loadtxt('./test_sim_bn_OFF_50comp/chain_1.txt')[:, :-4]
chain_off       = chain_off[len(chain_off)//4:]

chain_bn125     = np.loadtxt('./test_sim_bn_A125_50comp/chain_1.txt')[:, :-4]
chain_bn125     = chain_bn125[len(chain_bn125)//4:]

chain_bn125_8   = np.loadtxt('./test_sim_bn_A125_bnmodel/chain_1.txt')[:, :-4]
chain_bn125_8   = chain_bn125_8[len(chain_bn125_8)//4:]

chain_real      = np.loadtxt('./test_real_sv44/chain_1.txt')[:, :-4]
chain_real      = chain_real[len(chain_real)//4:]

chain_real_8par = np.loadtxt('./test_real_bn_bnmodel/chain_1.txt')[:, :-4]
chain_real_8par = chain_real_8par[len(chain_real_8par)//4:]

## Corner

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
    'J1640+2224_band_noise_gamma': 3.0,
    'J1640+2224_band_noise_log10_A': -12.5,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_8par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_8par.params]

fig = corner.corner(
    chain_gwbbn125_8par[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — GWB a -14 + band a -12.5, modello a 8 par',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_8par.png', dpi=150, bbox_inches='tight')

In [ ]:
chain_gwbbn125_8par_comuni = chain_gwbbn125_8par[:, [list(logL_gwbbn125_8par.params).index(p) for p in logL_gwbbn125_6par.params]]
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_6par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_6par.params]

# Primo corner
fig = corner.corner(
    chain_gwbbn125_6par[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_8par_comuni,
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#6A4C93',
    hist_kwargs={'color': '#6A4C93', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — GWB -14 + band -12.5: modello a 6 par (arancio) vs 8 par (viola)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_6par_vs_8par.png', dpi=150, bbox_inches='tight')

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
    'J1640+2224_band_noise_gamma': 3.0,
    'J1640+2224_band_noise_log10_A': -12.5,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_8par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_8par.params]

# Primo corner
fig = corner.corner(
    chain_bn125_8[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_8par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#6A4C93',
    hist_kwargs={'color': '#6A4C93', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 8 par: solo band -12.5 (arancio) vs GWB + band (viola)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_vs_solobn_8par.png', dpi=150, bbox_inches='tight')

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
    'J1640+2224_band_noise_gamma': 3.0,
    'J1640+2224_band_noise_log10_A': -12.5,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_8par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_8par.params]

# Primo corner
fig = corner.corner(
    chain_real_8par[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#2E86AB',
    hist_kwargs={'color': '#2E86AB', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_8par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 8 par: dati reali (blu) vs sim GWB + band (arancio)',
             fontsize=18, y=1.00)

fig.savefig('corner_reali_vs_sim_bn_A125_GWB_8par.png', dpi=150, bbox_inches='tight')

## Confronti a 6 par

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_6par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_6par.params]

# Primo corner
fig = corner.corner(
    chain_off[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_6par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#6A4C93',
    hist_kwargs={'color': '#6A4C93', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 6 par: senza GWB e senza band (arancio) vs GWB + band (viola)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_vs_OFF.png', dpi=150, bbox_inches='tight')

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_6par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_6par.params]

# Primo corner
fig = corner.corner(
    chain_gwboff_6par[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_6par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#6A4C93',
    hist_kwargs={'color': '#6A4C93', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 6 par: solo GWB (arancio) vs GWB + band (viola)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_vs_soloGWB.png', dpi=150, bbox_inches='tight')

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_6par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_6par.params]

# Primo corner
fig = corner.corner(
    chain_bn125[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_6par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#6A4C93',
    hist_kwargs={'color': '#6A4C93', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 6 par: solo band -12.5 (arancio) vs GWB + band (viola)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_vs_solobn125.png', dpi=150, bbox_inches='tight')

In [ ]:
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
truth_values = [truths_dict[p] for p in logL_gwbbn125_6par.params]
clean_labels = [name.replace('J1640+2224_', '') for name in logL_gwbbn125_6par.params]

# Primo corner
fig = corner.corner(
    chain_real[:],
    labels=clean_labels,
    truths=truth_values,
    truth_color='#222222',
    show_titles=True,
    title_fmt='.2f',
    label_kwargs={'fontsize': 14},
    title_kwargs={'fontsize': 12},
    color='#2E86AB',
    hist_kwargs={'color': '#2E86AB', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
)

# Secondo corner, sovrapposto
corner.corner(
    chain_gwbbn125_6par[:],
    labels=clean_labels,
    show_titles=False,            # non serve mostrare i titoli due volte
    color='#F4A261',
    hist_kwargs={'color': '#F4A261', 'linewidth': 1.5 , "density": True},
    fill_contours=True,
    levels=(0.68, 0.95, 0.997),
    smooth=1.0,
    smooth1d=None,
    bins=40,
    fig=fig,
    scale_hist=False,
);

fig.set_size_inches(16, 16)
fig.suptitle('J1640+2224 — modello a 6 par: dati reali (blu) vs sim GWB + band (arancio)',
             fontsize=18, y=1.00)

fig.savefig('corner_sim_bn_A125_GWB_vs_reali.png', dpi=150, bbox_inches='tight')

## Riepilogo

In [ ]:
# modello a 6 par: mediana e intervallo al 68% di ogni parametro
truths_dict = {
    'J1640+2224_dm_gp_gamma': 6.471070651479037,
    'J1640+2224_dm_gp_log10_A': -15.7227863246785,
    'J1640+2224_red_noise_gamma': 0.13235259553929019,
    'J1640+2224_red_noise_log10_A': -13.212902548624186,
    'J1640+2224_sv_gp_gamma': 1.3259815720503758,
    'J1640+2224_sv_gp_log10_A': -15.888300265911965,
}
etichette = ['reali', 'OFF', 'solo GWB', 'solo band', 'GWB+band']
catene = [chain_real, chain_off, chain_gwboff_6par, chain_bn125, chain_gwbbn125_6par]
colori = ['#2E86AB', '#F4A261', '#F4A261', '#F4A261', '#6A4C93']

fig, axs = plt.subplots(2, 3, figsize=(15, 8))
for ax, (i, name) in zip(axs.flat, enumerate(logL_gwbbn125_6par.params)):
    for j, (ch, col) in enumerate(zip(catene, colori)):
        q16, q50, q84 = np.percentile(ch[:, i], [16, 50, 84])
        ax.errorbar(j, q50, yerr=[[q50 - q16], [q84 - q50]], fmt='o', color=col, capsize=4)
    ax.axhline(truths_dict[name], color='#222222', lw=1)
    ax.set_xticks(range(len(etichette)))
    ax.set_xticklabels(etichette, rotation=30)
    ax.set_title(name.replace('J1640+2224_', ''), fontsize=12)
    ax.grid(alpha=0.25, linestyle='--')

fig.suptitle('Modello a 6 par: stessa realizzazione, cambiano solo GWB e band noise', fontsize=15)
fig.tight_layout()
fig.savefig('riepilogo_GWB_bn_A125_confronti.png', dpi=150, bbox_inches='tight')